In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
import os
os.environ["OPENAI_API_KEY"] = "5a4a5c93bdc34d8caac98deac76ecc82."
os.environ["GAODE_API_KEY"] = ""

In [3]:
llm = ChatOpenAI(
    model = "glm-4.6",
    openai_api_base="https://open.bigmodel.cn/api/paas/v4/",
    extra_body={
        "thinking": {"type": "disabled"} 
    }
)

## City 2 Adcode
- 高德API 中 需要城市的adcode 来查询
- City 2 Adcode excel表读取到内存中，省 市 区县 都有不同的 adcode，并且从属地关系 表示为顺序关系，例如山西省：12000，大同市：12001，同同区：12002
- 由于用户询问中的 位置 不完全等同于 映射表中的城市名字，第一步是需要大模型判断 用户位置 的三个级别：省 - 市 - 区县，

In [4]:
from pydantic import BaseModel
import pandas as pd
import requests

class WeatherAgent(BaseModel):
    adcode: str

class CityAdcode(BaseModel):
    province: str
    city: str
    district: str

data = pd.read_excel('/mnt/d/repo/agent/weather/AMap_adcode_citycode/AMap_adcode_citycode.xlsx')

@tool(args_schema=CityAdcode)
def query_adcode(province: str, city: str, district: str) -> str:
    """Query the adcode for a given province, city, and district."""
    tar = 0
    if province:
        for i in range(len(data)):
            if data['中文名'][i] == province:
                adcode = data['adcode'][i] 
                tar = i
                break
    if city:
        for i in range(tar, len(data)):
            if data['中文名'][i] == city:
                adcode = data['adcode'][i] 
                tar = i
                break
            if data['中文名'][i].endswith('省'):
                break 
    if district:
        for i in range(tar, len(data)):
            if data['中文名'][i] == district:
                adcode = data['adcode'][i] 
                tar = i
                break
            if data['中文名'][i].endswith('市'):
                break
    if 'adcode' not in locals():
        return ""
    return str(adcode)
@tool(args_schema=WeatherAgent) 
def get_weather(adcode: str) -> str:
    """Get the weather information for a given city."""
    if len(adcode) == 0:
        return f"City with adcode {adcode} not found."

    key = os.environ["GAODE_API_KEY"]
    url = f"https://restapi.amap.com/v3/weather/weatherInfo?city={adcode}&key={key}&extensions=base&output=JSON"

    response = requests.get(url)
    weather_data = response.json()
    return weather_data

In [5]:
agent = create_agent(
    model = llm,
    tools = [get_weather],
    system_prompt="""
    你是一个天气查询代理，负责根据用户的问题获取指定城市的天气信息。

    步骤：
    1. 分析用户查询中的地名，将其标准化为三级行政单位：省、市、区县。例如，用户说“石家庄长安区”，你需要判断为“河北省、石家庄市、长安区”。
    2. 使用“查询adcode”工具，提供这三个参数（省、市、区县）来从本地Excel表中获取正确的adcode。确保参数准确匹配表中的中文名。
    3. 使用“获取天气”工具，传入获取的adcode来调用高德API查询天气。
    4. 将天气信息以清晰、友好的方式反馈给用户。如果地名无法匹配或查询失败，说明原因。

    注意：地名必须是标准的三级名称；如果用户只提供部分信息（如仅区县），你需要推断完整的三级结构。始终使用工具，不要直接回答。
    """
)

In [6]:
response = agent.invoke(
    {"messages":[{"role":"user","content":"我现在想去杭州西湖游泳，天气如何？"}]}
)
print(len(response['messages']),response['messages'][-1].content)

4 
根据查询到的杭州西湖区天气信息：

🌤️ **当前天气状况：**
- **天气**：晴天
- **温度**：7°C
- **风向**：北风
- **风力**：≤3级
- **湿度**：45%
- **更新时间**：2025-11-19 18:32:30

⚠️ **游泳建议：**
目前温度只有7°C，天气虽然晴朗，但气温较低。在这样的温度下游泳会非常寒冷，不建议进行户外游泳活动。如果一定要游泳，建议：

1. 选择室内恒温游泳池
2. 如果坚持户外游泳，请务必做好充分的保暖准备
3. 游泳时间不宜过长
4. 有同伴陪同，注意安全

建议您等待气温回升后再考虑户外游泳活动。


In [ ]:
# 流式 按条 输出
for chunk in agent.stream(
    {"messages":[{"role":"user","content":"我现在想去杭州西湖游泳，天气如何？"}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'text', 'text': '\n我来帮您查询杭州西湖的天气情况。首先需要获取杭州地区的天气信息。\n'}, {'type': 'tool_call', 'name': 'get_weather', 'args': {'adcode': '330106'}, 'id': 'call_-8144064482968514692'}]
step: tools
content: [{'type': 'text', 'text': '{"status": "1", "count": "1", "info": "OK", "infocode": "10000", "lives": [{"province": "浙江", "city": "西湖区", "adcode": "330106", "weather": "多云", "temperature": "11", "winddirection": "西", "windpower": "≤3", "humidity": "24", "reporttime": "2025-11-19 13:32:30", "temperature_float": "11.0", "humidity_float": "24.0"}]}'}]
step: model
content: [{'type': 'text', 'text': '\n根据最新的天气信息，杭州西湖区目前的天气情况如下：\n\n🌤️ **天气状况**：多云\n🌡️ **温度**：11°C\n💨 **风向风力**：西风，≤3级\n💧 **湿度**：24%\n📅 **更新时间**：2025年11月19日 13:32\n\n**关于游泳的建议**：\n⚠️ **不太适合游泳** - 目前温度只有11°C，对于游泳来说水温太低，容易导致感冒或身体不适。\n\n如果您真想游泳，建议：\n1. 等待天气转暖，温度至少在20°C以上\n2. 选择室内恒温游泳池\n3. 关注天气预报，选择晴朗且温度较高的日子\n\n请注意保暖和安全！🏊\u200d♂️❄️'}]


In [ ]:
# 流式 按token 输出
for token, metadata in agent.stream(  
    {"messages": [{"role": "user", "content": "我现在想去杭州西湖游泳，天气如何？"}]},
    stream_mode="messages",
):
    # print(f"node: {metadata['langgraph_node']}")
    print(token.content_blocks)
    # break
    # if metadata['langgraph_node'] == 'model' and len(token.content_blocks) >= 1 and token.content_blocks[0]['type'] == 'text':
    #     print(token.content_blocks[0]['text'],end = '')

    # if metadata['langgraph_node'] == 'model':
    #     print(f"content: {token.content_blocks}")

[{'type': 'text', 'text': '\n'}]
[{'type': 'text', 'text': '我来'}]
[{'type': 'text', 'text': '帮'}]
[{'type': 'text', 'text': '您'}]
[{'type': 'text', 'text': '查询'}]
[{'type': 'text', 'text': '杭州'}]
[{'type': 'text', 'text': '西湖'}]
[{'type': 'text', 'text': '的'}]
[{'type': 'text', 'text': '天气'}]
[{'type': 'text', 'text': '情况'}]
[{'type': 'text', 'text': '。'}]
[{'type': 'text', 'text': '杭州'}]
[{'type': 'text', 'text': '西湖'}]
[{'type': 'text', 'text': '位于'}]
[{'type': 'text', 'text': '杭州市'}]
[{'type': 'text', 'text': '，'}]
[{'type': 'text', 'text': '我'}]
[{'type': 'text', 'text': '需要'}]
[{'type': 'text', 'text': '先'}]
[{'type': 'text', 'text': '获取'}]
[{'type': 'text', 'text': '该'}]
[{'type': 'text', 'text': '地区的'}]
[{'type': 'text', 'text': 'ad'}]
[{'type': 'text', 'text': 'code'}]
[{'type': 'text', 'text': '来'}]
[{'type': 'text', 'text': '查询'}]
[{'type': 'text', 'text': '天气'}]
[{'type': 'text', 'text': '信息'}]
[{'type': 'text', 'text': '。\n'}]
[{'type': 'tool_call_chunk', 'id': 'call_8fe490

In [60]:
async def stream_response(user_query: str):
    """Stream the agent's response for a given user query."""
    try:
        for token, metadata in agent.stream(  
            {"messages": [{"role": "user", "content": user_query}]},
            stream_mode="messages",
        ):
            if metadata['langgraph_node'] == 'model' and len(token.content) >= 1 and token.content_blocks[0]['type'] == 'text':
                text = token.content_blocks[0]['text']
                yield text
    except Exception as e:
        yield f"Error: {str(e)}"

In [65]:
import asyncio

async def main():
    async for chunk in stream_response("我现在想去杭州西湖游泳，天气如何？"):
        print(chunk, end='', flush=True)

await main()


我来帮您查询杭州西湖的天气情况。首先需要获取该地区的天气信息。

根据最新的天气信息，杭州西湖区目前的天气状况如下：

🌞 **天气**：晴朗
🌡️ **温度**：9°C
💨 **风向风力**：西风，≤3级
💧 **湿度**：41%
📅 **更新时间**：2025-11-19 17:05:21

⚠️ **游泳提醒**：
目前气温只有9°C，天气虽然晴朗，但温度较低，不太适合户外游泳。如果确实想游泳，建议：
- 选择室内游泳馆
- 如果一定要户外游泳，请务必做好保暖措施
- 注意水温较低可能对身体造成的影响

建议您等到气温回升后再考虑西湖游泳，或者选择其他室内游泳场所。